# 2026 TDL Challenge: GraphUniverse

This notebook evaluates your model across a grid of GraphUniverse's synthetic graph distributions:
- **Homophily** (low, mid, high)
- **Average degree** (low, high)  
- **Power-law exponent** (low, high)

Each configuration is trained with multiple random seeds. The best checkpoint from each run is then evaluated on all other grid settings (out-of-distribution evaluation).

## Setup Requirements

**Make sure Weights & Biases is configured:**
```bash
wandb login
```

## How to Use

1. Set your `MODEL_CONFIG` (path to your model yaml)
2. Run the evaluation
3. Results will be saved in:
   - `results.json` with detailed metrics
   - Heatmap visualizations showing performance across the grid
   - OOD performance delta plots

In [ ]:
import sys
from pathlib import Path

# Setup paths
_ROOT = Path.cwd().resolve()
_REPO = _ROOT if (_ROOT / "configs" / "run.yaml").exists() else _ROOT.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from utils import (
    resolve_project_root,
    run_challenge_grid,
    save_challenge_artifacts,
)

PROJECT_ROOT = resolve_project_root(_REPO)

## Configuration

**Set your model config path here:**

In [ ]:
# Your model configuration (e.g., "graph/gcn", "graph/gin", "graph/gat")
MODEL_CONFIG = "graph/gin"

## Sanity check

This cells check that you no cells below this point have been modified.

In [11]:
expected_hash = "631b2efaf4c84813b6bdb64ee1ed980f945f7fe71e4531aeb530fd1aae7bbbc3"

In [12]:
# UNIQUE_HASH_MARKER
import json
import hashlib

def hash_remaining_cells(notebook_path: str, marker_string: str = "# UNIQUE_HASH_MARKER") -> str:
    """
    Computes a SHA-256 hash of all cells from the marker cell to the end of the notebook.

    Args:
        notebook_path (str): The hardcoded path to the notebook file.
        marker_string (str, optional): A unique string present in the source of the calling cell.
                                       Defaults to "# UNIQUE_HASH_MARKER".

    Returns:
        str: The SHA-256 hash of the content of the target cells.

    Raises:
        ValueError: If the marker_string is not found in the notebook data.
    """
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_data = json.load(f)

    cells = notebook_data.get('cells', [])
    start_index = -1

    for i, cell in enumerate(cells):
        source = "".join(cell.get('source', []))
        if marker_string in source[:len(marker_string)]:
            start_index = i
            break

    if start_index == -1:
        raise ValueError(f"Marker string '{marker_string}' not found in the notebook.")

    content_to_hash = ""
    for cell in cells[start_index:]:
        content_to_hash += "".join(cell.get('source', []))

    return hashlib.sha256(content_to_hash.encode('utf-8')).hexdigest()

hash_value = hash_remaining_cells(notebook_path="run_evaluation.ipynb")

print(f"Computed hash: {hash_value}")

if hash_value != expected_hash:
    raise ValueError("❌ The content of the notebook has been modified. Please ensure that you are using the original notebook without any changes from this point onward.")
else:
    print("✅ Notebook content is verified.")


Computed hash: 631b2efaf4c84813b6bdb64ee1ed980f945f7fe71e4531aeb530fd1aae7bbbc3
✅ Notebook content is verified.


## Run Evaluation

This will train and evaluate your model across all grid configurations.

In [ ]:
results, study_id = run_challenge_grid(
    project_root=PROJECT_ROOT,
    model_config=MODEL_CONFIG,
)

## Save Results

Generate JSON output and visualization plots.

In [ ]:
output_paths = save_challenge_artifacts(
    results,
    model_config=MODEL_CONFIG,
    study_id=study_id,
)

print(f"\nResults saved to: {output_paths['dir']}")
print(f"JSON: {output_paths['json']}")

## Inspect Results

View results as a DataFrame for quick inspection.

In [ ]:
import pandas as pd

pd.DataFrame(results)